In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
from dtaianomaly.visualization import plot_demarcated_anomalies
from matplotlib.lines import Line2D

# NOTE: The raw anomaly scores for Chronos-Bolt and Time-MoE were padded with zeros instead of the first value of the scores.
# When calculating the metrics (VUS-PR, etc.), we applied post-processing to make the padding method consistent with the description in the paper (Figure 1), which is the first value of the scores.
# For visualization in this file the padding method makes negligible differences, so we sticked with the original raw scores.

# Normalization for better visualization
def normalize(array):
    return array / array.max()

In [ ]:
# Specify the file index for the TSB-AD-U evaluation set, which contains 350 files in total.
# e.g., file_ind = 0 corresponds to the first file in the evaluation set.
file_ind = 0

# Reconstruction-based variant for comparison
model_size = 'large' #can be 'base' or 'large'
win_size = 256 #can be 64, 256, 512, or 1024(only for 'large')

# Forecasting-based variant for comparison

forecasting_variant = 'timesfm-1.0-200m_64'
forecasting_label = 'TimesFM-1.0-64'

# Other forecasting-based variants for comparison (uncomment to use)

# forecasting_variant = 'TimeMoE-200M_512'
# forecasting_label = 'TimeMoE-large-512'

# forecasting_variant = 'bolt_base_256'
# forecasting_label = 'Chronos-Bolt-base-256'

# forecasting_variant = 'timesfm-2.0-500m_512'
# forecasting_label = 'TimesFM-2.0-512'

# Baseline for comparison with forecasting-based variant
baseline_variant = 'Last-3' #Can be 'Last-1', 'Last-3', or 'Centered-5'

data_direc = '../data/TSB-AD-U/'
file_name = pd.read_csv('../data/File_List/TSB-AD-U-Eva.csv')['file_name'].to_numpy()[file_ind]
file_name = os.path.splitext(file_name)[0]

score_direc_1 = f'../results/intermediate/moment_{model_size}_{win_size}/scores/{file_name}.npy'
score_direc_2 = f'../results/intermediate/{forecasting_variant}/scores/{file_name}.npy'
scores_1 = np.load(score_direc_1)
scores_2 = np.load(score_direc_2)

df = pd.read_csv(f'{data_direc}/{file_name}.csv').dropna()
X = df['Data'].to_numpy()
y = df['Label'].astype(int).to_numpy()
baseline_1 = df['Data'].rolling(window = win_size, center = True).var().bfill().ffill().to_numpy()

if baseline_variant == 'Last-1':
    baseline_2 = df['Data'].diff().bfill().ffill().to_numpy() ** 2
elif baseline_variant == 'Last-3':
    baseline_2 = (df['Data'].to_numpy() - df['Data'].rolling(window = 4, center = False).mean().bfill().ffill().to_numpy()) ** 2
elif baseline_variant == 'Centered-5':
    baseline_2 = (df['Data'].to_numpy() - df['Data'].rolling(window = 5, center = True).mean().bfill().ffill().to_numpy()) ** 2
else:
    raise ValueError('Invalid baseline variant')

plt.rcParams['font.size'] = 18

fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(5, figsize=(20, 12), sharex='all', height_ratios=[1] + 4*[0.5])
plot_demarcated_anomalies(X, y, ax=ax1)
plot_demarcated_anomalies(normalize(scores_1), y, ax=ax2)
plot_demarcated_anomalies(normalize(baseline_1), y, ax=ax3)
plot_demarcated_anomalies(normalize(scores_2), y, ax=ax4)
plot_demarcated_anomalies(normalize(baseline_2), y, ax=ax5)

legend_elements = [Line2D([0], [0], lw=2, label=f'MOMENT-{model_size}-{win_size}')]
ax2.legend(handles=legend_elements, loc='upper left', fontsize=18)

legend_elements = [Line2D([0], [0], lw=2, label=f'Var-{win_size}')]
ax3.legend(handles=legend_elements, loc='upper left', fontsize=18)

legend_elements = [Line2D([0], [0], lw=2, label=f'{forecasting_label}')]
ax4.legend(handles=legend_elements, loc='upper left', fontsize=18)

legend_elements = [Line2D([0], [0], lw=2, label=f'{baseline_variant}')]
ax5.legend(handles=legend_elements, loc='upper left', fontsize=18)

fontsize = 45
ax1.set_ylabel('Time series')
ax2.set_title(f'Reconstruction-based TSFM vs. Baseline', fontsize=fontsize)
ax2.set_ylabel(f'Anomaly score\n(normalized)')
ax3.set_ylabel(f'Anomaly score\n(normalized)')
ax4.set_title(f'Forecasting-based TSFM vs. Baseline', fontsize=fontsize)
ax4.set_ylabel(f'Anomaly score\n(normalized)')
ax5.set_ylabel(f'Anomaly score\n(normalized)')

fig.tight_layout()